# Análise de Performance - Verilator vs LUT2GPU

Este notebook processa dados de performance de implementações Verilator e LUT2GPU, usando o nome da pasta como dataset e gerando uma linha para cada arquivo CSV.

## Importando bibliotecas necessárias

In [63]:
import os
import pandas as pd
import numpy as np
from IPython.display import display, HTML

## Função para processar um único arquivo CSV

In [64]:
def process_single_csv(filepath, dataset_name):
    """
    Processa um único arquivo CSV e retorna suas informações.
    
    Args:
        filepath (str): Caminho do arquivo CSV
        dataset_name (str): Nome do dataset (pasta)
    
    Returns:
        pd.DataFrame: DataFrame com uma linha de resultados
    """
    try:
        df = pd.read_csv(filepath)
        
        # Identifica métodos disponíveis
        available_methods = df['implementation'].unique()
        
        # Prepara o DataFrame de resultado com valores padrão
        result_df = pd.DataFrame({
            'Timestamp': [df['timestamp'].min()],
            'Dataset': [dataset_name],
            'Inputs': [df['input_size'].unique()[0]],
            'Outputs': [df['output_size'].unique()[0]],
            'Combinations': [2**df['input_size'].unique()[0]],
            'Luts': 5
        })
        
        # Adiciona colunas para cada método
        methods_to_check = ['verilator','cpu', 'cuda', ]

        
        for method in methods_to_check:
            method_data = df[df['implementation'] == method] if method in available_methods else pd.DataFrame()
            
            if not method_data.empty:
                idx_min = method_data['execution_time_ms'].idxmin()
                if method == 'cuda':
                    result_df[f'{method.upper()}_Threads_Block'] = method_data.loc[idx_min, 'threads_per_block']
                    result_df[f'{method.upper()}_Blocks_Grid'] = method_data.loc[idx_min, 'blocks_per_grid']
                    result_df[f'{method.upper()}_Launches'] = method_data.loc[idx_min, 'launches']
                result_df[f'{method.upper()}_ExeTime_ms'] = method_data.loc[idx_min, 'execution_time_ms']
                result_df[f'{method.upper()}_Throughput_sec'] = method_data.loc[idx_min, 'throughput_combinations_per_sec']
            else:
                result_df[f'{method.upper()}_ExeTime_ms'] = '-'
                result_df[f'{method.upper()}_Throughput_sec'] = '-'
                if method == 'cuda':
                    result_df[f'{method.upper()}_Threads_Block'] = '-'
                    result_df[f'{method.upper()}_Blocks_Grid'] = '-'
                    result_df[f'{method.upper()}_Launches'] = '-'
        
        # Calcula diferença se ambos os métodos estiverem presentes e retorna quantas vezes mais rápido é o CUDA
        if 'cpu' in available_methods and 'cuda' in available_methods:
            result_df['CUDA_Speedup_CPU'] = result_df['CUDA_Throughput_sec'] / result_df['CPU_Throughput_sec']
        if 'verilator' in available_methods and 'cuda' in available_methods:
            result_df['CUDA_Speedup_Verilator'] = result_df['CUDA_Throughput_sec'] / result_df['VERILATOR_Throughput_sec']
        else:
            result_df['CUDA_Speedup'] = '-'
        
        return result_df
    
    except Exception as e:
        # print(f"Erro ao processar {filepath}: {e}")
        return None

## Função para processar todas as pastas

In [65]:
def process_all_folders(root_folder, not_got):
    """
    Processa todos os arquivos CSV em todas as pastas.
    
    Args:
        root_folder (str): Pasta raiz para busca
        not_got (list): Lista de pastas para não processar
    
    Returns:
        pd.DataFrame: DataFrame com todos os resultados
    """
    all_results = []
    
    for dirpath, dirnames, filenames in os.walk(root_folder):
        # Pega o nome da última pasta do caminho como dataset
        
        # Se a pasta ou subpasta estiver na lista de não processados, pula
        if any([ng in dirpath for ng in not_got]):
            continue
        
        dataset_name = os.path.basename(dirpath)
        
        for filename in filenames:
            if filename.endswith('.csv'):
                filepath = os.path.join(dirpath, filename)
                result = process_single_csv(filepath, dataset_name)
                
                if result is not None:
                    all_results.append(result)
    
    if not all_results:
        print("Nenhum arquivo CSV encontrado.")
        return None
    
    # Combina todos os resultados
    return pd.concat(all_results, ignore_index=True)

## Função para salvar resultados

In [66]:
def save_results(df, output_path='results/performance_summary.csv'):
    """
    Salva resultados em um arquivo CSV e exibe no notebook
    
    Args:
        df (pd.DataFrame): DataFrame com resultados
        output_path (str): Caminho para salvar o arquivo
    """
    if df is not None:
        df.to_csv(output_path, index=False)
        print(f"Resultados salvos em {output_path}")
        display(HTML(df.to_html(index=False)))

## Processando os dados

Agora vamos processar todos os arquivos CSV encontrados e gerar o relatório.

In [67]:
# Processa os arquivos em todas as pastas
root_folder = 'results'
not_got = ['results_until_26_02']
results = process_all_folders(root_folder, not_got)

if results is not None:
    save_results(results)

Resultados salvos em results/performance_summary.csv


Timestamp,Dataset,Inputs,Outputs,Combinations,Luts,VERILATOR_ExeTime_ms,VERILATOR_Throughput_sec,CPU_ExeTime_ms,CPU_Throughput_sec,CUDA_Threads_Block,CUDA_Blocks_Grid,CUDA_Launches,CUDA_ExeTime_ms,CUDA_Throughput_sec,CUDA_Speedup_CPU,CUDA_Speedup_Verilator
2025-02-26 22:35:27,26_02_c1355,41,32,2199023255552,5,3291281.510,326238.22,6836.905,4.577136e+08,256.0,768.0,1.0,1702.016,3.530132e+11,771.253452,1.082072e+06
2025-02-27 00:26:39,26_02_c17,5,2,32,5,1.610,19875.27,1.361,3.311347e+10,256.0,768.0,1.0,872.605,9.913709e+09,0.299386,4.987962e+05
2025-02-27 00:26:46,26_02_c1908,33,25,8589934592,5,3639886.980,294993.18,4131.606,6.414242e+08,256.0,768.0,1.0,1580.631,3.219104e+11,501.868102,1.091247e+06
2025-02-27 02:28:27,26_02_c3540,50,22,1125899906842624,5,7038522.790,152552.16,16567.851,4.400619e+08,256.0,768.0,1.0,1881.390,7.440504e+11,1690.785727,4.877351e+06
2025-02-27 06:23:47,26_02_c432,36,7,68719476736,5,2115459.628,507569.05,3908.123,3.626335e+08,256.0,768.0,1.0,1680.012,1.619664e+11,446.639386,3.191022e+05
2025-02-27 07:34:47,26_02_c499,41,32,2199023255552,5,2759658.273,389085.07,15879.721,1.968075e+08,256.0,768.0,1.0,1760.273,3.408833e+11,1732.064525,8.761150e+05
2025-02-27 09:07:32,26_02_c5315,178,123,0,5,4926446.692,217954.62,422.195,2.028621e+10,256.0,768.0,1.0,1690.665,9.726524e+11,47.946478,4.462637e+06


In [68]:
if results is not None:
    # Mostra todas as colunas
    pd.set_option('display.max_columns', None)
    display(results)

,Timestamp,Dataset,Inputs,Outputs,Combinations,Luts,VERILATOR_ExeTime_ms,VERILATOR_Throughput_sec,CPU_ExeTime_ms,CPU_Throughput_sec,CUDA_Threads_Block,CUDA_Blocks_Grid,CUDA_Launches,CUDA_ExeTime_ms,CUDA_Throughput_sec,CUDA_Speedup_CPU,CUDA_Speedup_Verilator
0,2025-02-26 22:35:27,26_02_c1355,41,32,2199023255552,5,3291281.510,326238.22,6836.905,4.577136e+08,256.0,768.0,1.0,1702.016,3.530132e+11,771.253452,1.082072e+06
1,2025-02-27 00:26:39,26_02_c17,5,2,32,5,1.610,19875.27,1.361,3.311347e+10,256.0,768.0,1.0,872.605,9.913709e+09,0.299386,4.987962e+05
2,2025-02-27 00:26:46,26_02_c1908,33,25,8589934592,5,3639886.980,294993.18,4131.606,6.414242e+08,256.0,768.0,1.0,1580.631,3.219104e+11,501.868102,1.091247e+06
3,2025-02-27 02:28:27,26_02_c3540,50,22,1125899906842624,5,7038522.790,152552.16,16567.851,4.400619e+08,256.0,768.0,1.0,1881.390,7.440504e+11,1690.785727,4.877351e+06
4,2025-02-27 06:23:47,26_02_c432,36,7,68719476736,5,2115459.628,507569.05,3908.123,3.626335e+08,256.0,768.0,1.0,1680.012,1.619664e+11,446.639386,3.191022e+05
5,2025-02-27 07:34:47,26_02_c499,41,32,2199023255552,5,2759658.273,389085.07,15879.721,1.968075e+08,256.0,768.0,1.0,1760.273,3.408833e+11,1732.064525,8.761150e+05
6,2025-02-27 09:07:32,26_02_c5315,178,123,0,5,4926446.692,217954.62,422.195,2.028621e+10,256.0,768.0,1.0,1690.665,9.726524e+11,47.946478,4.462637e+06
